In [ ]:
import pandas as pd
from typing import Optional, Dict
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.filtering import apply_filters, only_korean, has_value, FilterValue

In [ ]:
# NaN 체크 함수
#1. 간단한 NaN 체크
na = lambda df, col: df[col].isna().sum()

#2. NaN 비율 체크
def check_na(df, col):
    total = len(df)
    na = df[col].isna().sum()
    not_na = df[col].count()
    
    print(f"[{col}]")
    print(f"전체: {total}")
    print(f"NaN: {na}")
    print(f"Not NaN: {not_na}")
    print(f"NaN 비율: {na / total:.2%}")
#check_na(df_docu, '매체')

# 3. 여러 컬럼에 대해 NaN 체크
def check_na_df(df, cols):
    result = []
    
    for col in cols:
        total = len(df)
        na = df[col].isna().sum()
        not_na = df[col].count()
        
        result.append({
            "column": col,
            "total": total,
            "na": na,
            "not_na": not_na,
            "na_ratio": na / total
        })
    
    return pd.DataFrame(result)
#check_na_df(df_docu, ['매체', 'file_id'])

In [ ]:
# 컬럼 순서 재배치
front_cols = ['ID','sen_id',"speaker", 'sentence_form']
last_cols = ['category', '매체', '파일제목', '출판사', 'head','docu_lenth',
       'dominant_EN_No', 'dominant_count', 'dominant_ratio' ]
rest_cols = [col for col in out.columns if col not in front_cols + last_cols]
out = out[front_cols + rest_cols + last_cols]

In [ ]:
# 컬럼명 중복 제거(_x, _y)
y_cols = [c for c in out.columns if c.endswith("_y")]
out = out.drop(columns=y_cols)

out.columns = [c[:-2] if c.endswith("_x") else c for c in out.columns]

In [ ]:
#form frequency 계산
Columns_to_check =['V0_No', 'V0_label', 'V0_form']#['f_EN_label','f_EN_form', 'f_EN_No']# ['VX0_No', 'VX0_form']
form_freq = (
    df.groupby(Columns_to_check)['count']
      .sum()
      .reset_index()
      .sort_values('count', ascending=False)
)

form_freq.to_csv(r'C:\Users\yu2hy\OneDrive\◎2020_copus\python_2023\label\V0_freq.csv', index=False, encoding="utf-8-sig")

In [ ]:
# ---
# document 정보에서 필요한 컬럼 병합
# ---
df_word = df.copy()
def safe_merge(df, df_add, key, cols):
    cols_to_use = [key] + [c for c in cols if c not in df.columns]
    return df.merge(df_add[cols_to_use], on=key, how="left")

df_word = df.copy()

df = safe_merge(df, df_sen, "sen_id", ["speaker"])
df = safe_merge(
    df,
    df_docu,
    "docu_id",
    ['category', '매체', '내용', '파일제목', '저자', '출판사', '출판연도',
     'head', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio']
)

print(f"merge 완료 - {stamp}")

In [ ]:
SEN_ID = "EO0307.06078"

df_sen.loc[df_sen["sen_id"] == SEN_ID, ["sen_id", "sentence_form"]]


In [ ]:
# Next_VX_No, vx0_No의 800을 801로 변경
df[["Next_VX_No", "vx0_No"]] = df[["Next_VX_No", "vx0_No"]].replace(800, 801)

In [ ]:
df_word[df_word["Next_VX_No"] == 800].count()

### VX가 없는 경우, word_id2로 V_word_id, f_word_id 채우기

mask = (df_word["V_form"].notna()) & (df_word["V_word_id"] == -1)
df_word.loc[mask, "V_word_id"] = df_word.loc[mask, "word_id2"]
mask = (df_word["EN_form"].notna()) & (df_word["f_word_id"] == -1)
df_word.loc[mask, "f_word_id"] = df_word.loc[mask, "word_id2"]
print(f"완료 - {stamp}")

In [ ]:
# 컬럼명 변경
df = df.rename(columns={
    "파일제목": "file_title",
    "출판연도": "pub_year"
})

In [ ]:
# 컬럼 만들어서 값 넣기
#둘 다 T이거나 둘 다 F면 True, → 하나만 T면 False
df["시제_유지"] = df["sentence_f_EP_T"] == df["prev_sentence_f_EP_T"]

In [ ]:
#df[df['f_word_id']==-1].count()
#df[df['V_form']=='']['vx_len'].unique()
print(df[(df['vx_len']==0) & (df['vx0_form'].isna())]['f_word_id'].unique())
mask = (df["vx_len"] == 0) & (df["vx0_form"].notna())
(df.loc[mask, "f_word_id"] == df.loc[mask, "word_id2"]).all()

In [ ]:
#값 교체
df_word[["Next_VX_No", "vx0_No"]] = df_word[["Next_VX_No", "vx0_No"]].replace(800, 801)

df.loc[df["문서범주"] == "책", "문서범주"] = "인문자연"

In [ ]:
# 조건 넣어 값 확인.
df.loc[df['f_EN_No'] == 1101, 'EN_No']
df.loc[df['f_EN_No'] == 1101, 'EN_No'].unique()
df.loc[df['f_EN_No'] == 1101, 'EN_No'].value_counts()

In [ ]:
# 문장 파일 저장 컬럼 순서 재배치
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")


front_cols = ['file_id', 'docu_id', 'sen_id', 'file_sen_num', 'sen_num',
       'rev_sen_num']
last_cols = ['quote_count', 'quote_positions', 'quote_group_id', 'quote_span_len', 'long_quote', 'quote_start_sen_id', 'quote_end_sen_id', 'span_wrap_ok', 'span_type', 'passage_type', 'review_flag', 'review_note', 'quote_type_fix', 'modified_at', 'quote_end_offset']
rest_cols = [col for col in df_sen.columns if col not in front_cols + last_cols]
df_sen = df_sen[front_cols + rest_cols + last_cols]


sen_out_path = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.7_sen.csv"    
df_sen.to_csv(sen_out_path, index=False, encoding= "utf-8")
print(f"{sen_out_path}로 저장했습니다.")
print(f"완료 - {stamp}")

In [ ]:
# 저장하기
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")

output_path = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.6.csv"
df_word.to_csv(output_path, index=False, encoding= "utf-8")
print(f"{output_path}로 저장했습니다.")
print(f"완료 - {stamp}")

In [ ]:
# 컬럼명 변경
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H-%M-%S")

df_word = df_word.rename(columns={
    'word_id2': "word_id",
    'rev_word_id2': "rev_word_id",
    "V_form_0": "V_form",
    "V_label_0": "V_label",
    'V_word_id': "V0_word_id",
    'vx_len': "VX_len",
    'vx0_No': "VX0_No",
    'vx0_form': "VX0_form",
    'vx0_order': "VX0_order",
})

print(f"컬럼 변경 완료 - {stamp}")
df_word.columns